# Compare Explanation Families

Stage 2 asks whether automated simulatability distinguishes explanation families from no-explanation baselines under the harmonized one-sample-per-prompt setup.

The previous working notebook was archived to `notebooks/old/5_compare_families.ipynb` for reference.

What this notebook does:

1. Load one judge's v2 score CSV.
2. Infer each row's explanation family.
3. Canonicalize duplicated baseline rows by taking their mean.
4. Summarize score distributions with the same 10-seed bucket convention used elsewhere.
5. Select the best non-baseline method/configuration per family.
6. Plot best family configs against baselines, separated by dataset.
7. Build a best-method allow-list for the future `data/best_prompts/` subset.

Reusable visualization code lives in `utils/plot.py`. This notebook keeps data filtering and selection logic explicit for traceability.


## 1. Parameters

Change these values to control the scope. Defaults target the Qwen3.5-9B Stage 2 comparison.


In [ ]:
MODEL_FILE = "Qwen_Qwen3.5-9B"

# Stage 2 currently compares the harmonized current ConSim concept prompts
# with rationale and attribution prompts. If Stage 1 selects simulator_consim,
# change this list after regenerating and rescoring the corresponding prompts.
SPECIFICATIONS = ["new_consim", "rationales", "attributions"]

# Optional filters. None means keep everything present in the score file.
DATASETS = None             # e.g. ["RT", "AG"]
CLASSES_SUBSETS = None      # e.g. ["[0, 1]"]
FAMILIES = None             # e.g. ["concepts", "rationales", "attributions"]
PROMPT_TYPES = None         # e.g. ["B1", "B2", "C1", "R1", "A1"]
METHODS = None              # e.g. ["SemiNMF", "saliency"]

# Notebook 5 is intentionally non-anonymized only.
INCLUDE_ANONYMIZED = False

# Optional manual best-config override. Keep None to select best configs
# automatically from complete, non-corrupted candidates. When specified, use
# family names as keys and family_config strings as values, for example:
# BEST_CONFIGS = {"concepts": "SemiNMF / topk", "rationales": "Qwen/Qwen3.5-2B", "attributions": "saliency"}
BEST_CONFIGS = None

# Bucket convention used throughout this repo: 50 seeds -> 5 groups of 10.
SEEDS_PER_BUCKET = 10
EXPECTED_SEEDS = 50

# Paper/export controls. Keep both False while iterating.
EXPORT_FIGURES = False
EXPORT_ALLOW_LIST = False


## 2. Imports and Paths


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.plot import plot_pairwise_difference_matrix, plot_ranked_score_bars
from utils.analysis import (
    EXCLUDED_BEST_METHODS,
    FAMILY_COLORS,
    FAMILY_ORDER,
    NON_ANON_BASELINES,
    NON_ANON_PROMPT_TYPES,
    NON_ANON_PROMPT_TYPES_BY_FAMILY,
    add_contender,
    add_family_columns,
    best_configs_by_family,
    canonicalize_baselines,
    complete_candidate_configs,
    filter_keep,
    grouped_bucket_stats,
    keep_ge_class_subset_len,
    random_chance_for,
)

DATA_DIR = REPO_ROOT / "data"
EXPORT_DIR = REPO_ROOT / "LaTeX-Simulatability-Shortcut" / "plots"
BEST_PROMPTS_DIR = DATA_DIR / "best_prompts"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PROMPTS_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load Scores

Stage 2 uses v2 scores because they include parser diagnostics and invalid-output handling.


In [ ]:
CSV_PATH = DATA_DIR / f"consim_{MODEL_FILE}_v2.csv"
assert CSV_PATH.exists(), f"Score CSV not found: {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows from {CSV_PATH.relative_to(REPO_ROOT)}")
print(f"datasets: {sorted(df['dataset'].dropna().unique())}")
print(f"specifications: {sorted(df['specification'].dropna().unique())}")
print(f"prompt_types: {sorted(df['prompt_type'].dropna().unique())}")
print(f"methods: {sorted(map(str, df['method'].dropna().unique()))[:30]}")
df.head()


## 4. Family Inference and Shared Helpers

The score CSV has no explicit `family` column. We infer it from prompt type first, then method name as a fallback.


In [ ]:
ROW_KEY_COLS = ["dataset", "model", "classes_subset", "seed", "method", "nb_concepts", "interpretation", "prompt_type", "specification"]
BASELINE_KEY_COLS = ["dataset", "model", "classes_subset", "seed", "prompt_type"]


## 5. Prepare Analysis Table

We keep selected explanation families plus shared baselines. Baselines are canonicalized before any aggregation.


In [ ]:
df = add_family_columns(df)

print("Family inference summary:")
display(df.groupby(["family_raw", "prompt_type", "method"]).size().reset_index(name="n").sort_values(["family_raw", "prompt_type", "method"]))

selected_families = FAMILY_ORDER if FAMILIES is None else list(FAMILIES)
df_f = filter_keep(
    df,
    keep={
        "dataset": DATASETS,
        "classes_subset": CLASSES_SUBSETS,
        "specification": SPECIFICATIONS,
        "prompt_type": PROMPT_TYPES,
        "method": METHODS,
        "family_raw": set(selected_families) | {"baseline"},
    },
)
df_f = keep_ge_class_subset_len(df_f, 3)

dropped_anonymized = len(df_f) - len(df_f[df_f["prompt_type"].isin(NON_ANON_PROMPT_TYPES)])
if dropped_anonymized:
    print(f"Dropped anonymized prompt-type rows: {dropped_anonymized:,}")
df_f = filter_keep(df_f, keep={"prompt_type": NON_ANON_PROMPT_TYPES})
assert set(df_f["prompt_type"].dropna().unique()).issubset(NON_ANON_PROMPT_TYPES)

df_f = canonicalize_baselines(
    df_f,
    row_key_cols=ROW_KEY_COLS,
    baseline_key_cols=BASELINE_KEY_COLS,
    baseline_mask=df_f["family_raw"] == "baseline",
    family_raw_col="family_raw",
)
print(f"Rows after filtering and baseline handling: {len(df_f):,}")
display(df_f.groupby(["family", "prompt_type", "specification"]).size().reset_index(name="n"))


## 6. Overall Family/Prompt-Type Summary

This table is a broad diagnostic. The paper figures below use the best config per family rather than pooling all methods.


In [ ]:
SUMMARY_COLS = ["dataset", "family", "prompt_type"]
summary_stats = grouped_bucket_stats(
    df_f,
    SUMMARY_COLS,
    seeds_per_bucket=SEEDS_PER_BUCKET,
    expected_seeds=EXPECTED_SEEDS,
)
summary_stats.sort_values(["dataset", "family", "prompt_type"])


## 7. Select the Best Method/Configuration per Family

If `BEST_CONFIGS` is set in the parameter cell, this section uses those configs directly. Otherwise, best configs are selected globally over the current filters and non-anonymized prompt types. Automatic selection excludes `ClassesAs` and any candidate with incomplete or corrupted coverage.


In [ ]:
candidate_prompt_types = set().union(*NON_ANON_PROMPT_TYPES_BY_FAMILY.values())
best_candidates = df_f.copy()
best_candidates = best_candidates[best_candidates["family"].isin(FAMILY_ORDER)].copy()
best_candidates = best_candidates[best_candidates["prompt_type"].isin(candidate_prompt_types)].copy()
best_candidates = best_candidates[~best_candidates["method"].astype(str).isin(EXCLUDED_BEST_METHODS)].copy()
best_candidates = best_candidates[
    best_candidates.apply(lambda row: row["prompt_type"] in NON_ANON_PROMPT_TYPES_BY_FAMILY.get(row["family"], set()), axis=1)
]

BEST_CONFIG_COLS = ["family", "family_config"]
complete_candidates, incomplete_coverage = complete_candidate_configs(best_candidates, expected_seeds=EXPECTED_SEEDS)
if not incomplete_coverage.empty:
    print("Incomplete candidates excluded from automatic best-method selection:")
    display(incomplete_coverage)

if BEST_CONFIGS is not None:
    # Manual override: do not compute best configs. This is useful once the
    # paper's Stage 3 allow-list is fixed. Values are family_config strings.
    best_configs = pd.DataFrame(
        [{"family": family, "family_config": config} for family, config in BEST_CONFIGS.items()]
    )
    best_config_stats = pd.DataFrame(columns=BEST_CONFIG_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])
    selected_candidate_rows = best_candidates
    print("Using manual BEST_CONFIGS override; automatic ranking skipped.")
else:
    selected_candidate_rows = complete_candidates
    best_configs, best_config_stats = best_configs_by_family(
        complete_candidates,
        seeds_per_bucket=SEEDS_PER_BUCKET,
        expected_seeds=EXPECTED_SEEDS,
    )

print("Best config per family:")
display(best_configs)
print("Complete candidate configs used for automatic ranking:")
display(best_config_stats)


## 8. Best Family Configs vs Baselines

These are the main Stage 2 plots. Each panel is one dataset. GE is restricted to class subsets of length 3.


In [ ]:
if best_configs.empty:
    best_rows = selected_candidate_rows.iloc[0:0].copy()
else:
    best_keys = best_configs[BEST_CONFIG_COLS].drop_duplicates()
    best_rows = selected_candidate_rows.merge(best_keys, on=BEST_CONFIG_COLS, how="inner")
    missing_manual = set(map(tuple, best_keys.to_numpy())) - set(map(tuple, best_rows[BEST_CONFIG_COLS].drop_duplicates().to_numpy()))
    if missing_manual:
        print(f"warning: selected configs have no matching rows after filters: {sorted(missing_manual)}")

PLOT_GROUP_COLS = ["dataset", "family", "family_config", "prompt_type"]
best_expl_stats = grouped_bucket_stats(
    best_rows,
    PLOT_GROUP_COLS,
    seeds_per_bucket=SEEDS_PER_BUCKET,
    expected_seeds=EXPECTED_SEEDS,
) if not best_rows.empty else pd.DataFrame(columns=PLOT_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])

baseline_rows = df_f[df_f["family"] == "baseline"].copy()
baseline_rows["family_config"] = "baseline"
baseline_stats = grouped_bucket_stats(
    baseline_rows,
    PLOT_GROUP_COLS,
    seeds_per_bucket=SEEDS_PER_BUCKET,
    expected_seeds=EXPECTED_SEEDS,
) if not baseline_rows.empty else pd.DataFrame(columns=PLOT_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])

best_plot_stats = pd.concat([best_expl_stats, baseline_stats], ignore_index=True).dropna(subset=["mean_score"])
best_plot_stats["label"] = (
    best_plot_stats["family"].astype(str)
    + "\n" + best_plot_stats["family_config"].astype(str).str.slice(0, 28)
    + "\n" + best_plot_stats["prompt_type"].astype(str)
)
best_plot_stats.sort_values(["dataset", "mean_score"], ascending=[True, False])


In [ ]:
for panel, sub in best_plot_stats.groupby("dataset", dropna=False):
    safe_panel = str(panel).replace(" ", "").replace("|", "_").replace("/", "-")
    ax = plot_ranked_score_bars(
        sub,
        label_col="label",
        value_col="mean_score",
        err_col="std_score",
        color_col="family",
        color_map=FAMILY_COLORS,
        ylabel="Simulatability score",
        title=(
            f"Best family configs + baselines | {panel} | judge={MODEL_FILE}\n"
            f"ranked highest to lowest; error bars = std over {SEEDS_PER_BUCKET}-seed buckets"
        ),
        random_chance=random_chance_for(df_f[df_f["dataset"] == panel]),
        save_dir=EXPORT_DIR if EXPORT_FIGURES else None,
        file_name=f"families_best_{safe_panel}.pdf" if EXPORT_FIGURES else None,
    )
    plt.show()


## 9. Pairwise Prompt-Type Differences

These matrices compare the selected best family prompt types and baselines on common dataset/class-subset/seed cells. Only the score-difference matrix is shown; bold cells are significant paired t-tests.


In [ ]:
pairwise_parts = []
if not best_rows.empty:
    expl_pairwise = add_contender(best_rows, parts=["family", "family_config", "prompt_type"])
    pairwise_parts.append(expl_pairwise)

if not baseline_rows.empty:
    baseline_pairwise = baseline_rows.copy()
    baseline_pairwise["contender"] = "baseline / " + baseline_pairwise["prompt_type"].astype(str)
    pairwise_parts.append(baseline_pairwise)

prompt_pairwise_df = pd.concat(pairwise_parts, ignore_index=True) if pairwise_parts else pd.DataFrame()
print(f"Pairwise prompt-type rows: {len(prompt_pairwise_df):,}")
print(f"Contenders: {sorted(prompt_pairwise_df['contender'].dropna().unique()) if not prompt_pairwise_df.empty else []}")


In [ ]:
if prompt_pairwise_df.empty:
    print("No rows available for prompt-type pairwise matrices.")
else:
    pairwise_index = ["dataset", "model", "classes_subset", "seed", "contender"]
    matrix_specs = [(dataset, sub.copy()) for dataset, sub in prompt_pairwise_df.groupby("dataset", dropna=False)]
    matrix_specs.append(("All datasets", prompt_pairwise_df.copy()))

    prompt_difference_matrices = {}
    for panel, sub in matrix_specs:
        safe_panel = str(panel).replace(" ", "").replace("|", "_").replace("/", "-")
        prompt_pairwise_scores = sub.set_index(pairwise_index)["score"]
        fig_diff_prompt, prompt_difference_matrix = plot_pairwise_difference_matrix(
            prompt_pairwise_scores,
            compared_index="contender",
            title_prefix=f"Best family prompt types + baselines | {panel} | judge={MODEL_FILE}",
            figsize=(7, 5.5),
        )
        if EXPORT_FIGURES:
            fig_diff_prompt.savefig(EXPORT_DIR / f"families_prompt_type_pairwise_diff_{safe_panel}.pdf")
        plt.show()
        prompt_difference_matrices[panel] = prompt_difference_matrix
        display(prompt_difference_matrix)


## 10. Best-Prompt Allow-List for Stage 3

This allow-list is meant for the planned `scripts/make_best_prompts.py`. It records selected best methods/configurations and matching baselines, without rewriting any prompt keys.


In [ ]:
def config_rows_for_allow_list(rows: pd.DataFrame, family: str) -> list[dict]:
    family_rows = rows[rows["family"] == family]
    out = []
    for _, row in family_rows.drop_duplicates(["method", "interpretation", "family_config"]).iterrows():
        entry = {"method": row["method"], "family_config": row["family_config"]}
        if family == "concepts" and not pd.isna(row.get("interpretation")):
            entry["interpretation"] = row["interpretation"]
        out.append(entry)
    return out

allow_list = {
    "concepts": config_rows_for_allow_list(best_rows, "concepts"),
    "rationales": config_rows_for_allow_list(best_rows, "rationales"),
    "attributions": config_rows_for_allow_list(best_rows, "attributions"),
    "baselines": sorted(NON_ANON_BASELINES if not INCLUDE_ANONYMIZED else (NON_ANON_BASELINES | ANON_BASELINES)),
    "source_score_file": str(CSV_PATH.relative_to(REPO_ROOT)),
    "notes": "Methods selected by notebooks/5_compare_families.ipynb; prompt keys must be preserved when copying rows to data/best_prompts/.",
}

print(json.dumps(allow_list, indent=2))

if EXPORT_ALLOW_LIST:
    out_path = BEST_PROMPTS_DIR / "allow_list.json"
    out_path.write_text(json.dumps(allow_list, indent=2) + "\n")
    print(f"Wrote {out_path.relative_to(REPO_ROOT)}")


## 10. Paper Notes

After running the notebook, record the paper-facing takeaway here:

- Which best method/configuration was selected for each family?
- Are explanation-family scores visually separated from the baselines?
- Which plots should be included in `LaTeX-Simulatability-Shortcut/main.tex`?
- Should Stage 3 proceed with this allow-list, or should Stage 1's `simulator_consim` decision change the prompt set first?
